# FIGURE 2 — simulated cassette splicing: load, PCA, validate

Loads the simulated data from `pipelines/09_simulation/00_simulate_splicing.py` and checks it
behaves as designed **before** any model is run on it.

**The simulation** (see `METHODS_simulation.md`): cassette exon-skipping events only (3 junctions:
two inclusion + one skip), conditioned on *real* ATSE coverage from mouse heart, with counts drawn
from a **Binomial**. Ground truth is known exactly:

| field | meaning |
|---|---|
| `obs['true_program']` | which of K programs each cell was assigned to |
| `obsm['true_phi']` | (N, K) program mixture per cell |
| `uns['true_psi_skip']` | (K, n_ATSE) skip usage per program |
| `var['is_differential']` | ATSE truly differs between programs (calibration label) |
| `uns['atse_delta_psi']` | max−min skip usage across programs = effect size |

**Two datasets:** `eff1.0` (30% of cassettes truly differential) and `eff0.0` (**pure null** — no
cassette differs between programs; any structure PCA finds there is noise, so it is the false-positive
control).

**Why centred PSI:** we PCA `junc_ratio_centered`, produced by `leafletfa.waypoints.calculate_centered_psi`
— the *same* function the real pipeline uses. It is **not** a naive mean-subtraction: each PSI is
weighted by the beta-binomial variance implied by its ATSE coverage, so a PSI from 2 reads is
down-weighted against one from 200. Using the pipeline's own centring keeps this PCA comparable to
PCA on real data.

In [ ]:
import json

import anndata as ad
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import TruncatedSVD
from scipy.stats import pearsonr

plt.rcParams["figure.dpi"] = 110

SIM = "/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/SIMULATION"
SIGNAL = f"{SIM}/sim_heart_K5_binom_eff1.0_seed0.h5ad"   # 30% cassettes differential
NULL   = f"{SIM}/sim_heart_K5_binom_eff0.0_seed0.h5ad"   # pure null (FPR control)

a = ad.read_h5ad(SIGNAL)
params = json.loads(a.uns["sim_params"])
K = params["K"]

print(f"{a.n_obs:,} cells x {a.n_vars:,} junctions ({a.n_vars//3:,} cassettes)")
print("layers:", list(a.layers))
print("params:", {k: params[k] for k in ["tissue", "K", "emission", "frac_differential", "effect", "seed"]})
print("\ncells per program:", np.bincount(a.obs['true_program']))
print("junction roles:", a.var['junction_role'].value_counts().to_dict())

## 1. Does the data recover the ground truth?

Before PCA: pool cells **by their true program** and re-estimate the skip usage of each cassette from
the simulated counts. This should return the `true_psi_skip` we put in.

Programs are unbalanced (they are seeded from real heart cell types), so per-program PSI is noisy for
the small ones — we therefore report recovery as a function of how many cells actually cover the ATSE.

In [ ]:
def recover_psi(adata):
    """Empirical skip usage per (program, cassette) from the simulated counts."""
    J = sp.csr_matrix(adata.layers["cell_by_junction_matrix"]).toarray()
    C = sp.csr_matrix(adata.layers["cell_by_cluster_matrix"]).toarray()
    ev = adata.var["event_id"].astype(str).values
    events = np.asarray(adata.uns["atse_ids"], dtype=object)
    skip = np.where(adata.var["junction_role"].values == "J3")[0]
    prog = adata.obs["true_program"].values
    Kk = adata.obsm["true_phi"].shape[1]

    emp = np.full((Kk, len(events)), np.nan)
    ncell = np.zeros((Kk, len(events)))
    for k in range(Kk):
        cells = np.where(prog == k)[0]
        num = J[np.ix_(cells, skip)].sum(0)
        den = C[np.ix_(cells, skip)].sum(0)
        ncell[k] = (C[np.ix_(cells, skip)] > 0).sum(0)
        emp[k] = np.where(den > 0, num / np.maximum(den, 1), np.nan)
    return emp, ncell


emp, ncell = recover_psi(a)
true = np.asarray(a.uns["true_psi_skip"])
ev = a.var["event_id"].astype(str).values
events = np.asarray(a.uns["atse_ids"], dtype=object)
ei = {e: i for i, e in enumerate(events)}
col_ev = np.array([ei[e] for e in ev])
is_diff_atse = np.array([a.var["is_differential"].values[col_ev == i][0] for i in range(len(events))])

print("recovery of true skip usage (Pearson r), by minimum cells covering the ATSE:")
for m in [0, 20, 100]:
    ok = (ncell >= m) & ~np.isnan(emp)
    print(f"  >= {m:>3} cells: r = {pearsonr(true[ok], emp[ok])[0]:.3f}   (n = {ok.sum():,})")

In [ ]:
# truth vs recovered, and the differential/null separation
ok = (ncell >= 100) & ~np.isnan(emp)

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].scatter(true[ok], emp[ok], s=3, alpha=0.2, rasterized=True)
ax[0].plot([0, 1], [0, 1], "r--", lw=1)
ax[0].set_xlabel("TRUE skip usage"); ax[0].set_ylabel("recovered from counts")
ax[0].set_title(f"Ground truth recovered\nPearson r = {pearsonr(true[ok], emp[ok])[0]:.3f}")

e2 = np.where(ncell >= 100, emp, np.nan)
with np.errstate(all="ignore"):
    delta = np.nanmax(e2, 0) - np.nanmin(e2, 0)
good = np.isfinite(delta)
sns.boxplot(x=np.where(is_diff_atse[good], "differential", "null"), y=delta[good], ax=ax[1])
ax[1].set_ylabel("empirical Δ skip usage (max − min across programs)")
ax[1].set_xlabel("")
ax[1].set_title("Differential cassettes separate from nulls\n(nulls should sit at ~0)")
plt.tight_layout(); plt.show()

print(f"median Δ  differential: {np.nanmedian(delta[good & is_diff_atse]):.3f}")
print(f"median Δ  null        : {np.nanmedian(delta[good & ~is_diff_atse]):.3f}  <- should be ~0")

### 1b. Worked example — one differential cassette, junction by junction

Pick a cassette the simulator *planted* as differential, and show the per-cell PSI of **each of its
three junctions** across the programs it was made to differ in.

What to look for:
- **J3 (skip)** and the two **inclusion** junctions (J1, J2) must move in **opposite directions** —
  the three probabilities sum to 1 within the event, so more skipping necessarily means less inclusion.
- The observed per-cell PSI should scatter around the **planted truth** (red diamonds).
- The **null** cassette shown alongside should look flat across programs — same PSI everywhere.


In [ ]:
# ── pick a differential cassette (large planted effect, well covered) and a null one ──
Cd = sp.csr_matrix(a.layers["cell_by_cluster_matrix"]).toarray()
Rd = sp.csr_matrix(a.layers["junc_ratio"]).toarray()
role_all = a.var["junction_role"].values
prog_all = a.obs["true_program"].values
true_skip = np.asarray(a.uns["true_psi_skip"])          # (K, n_ATSE)
delta_true_all = np.asarray(a.uns["atse_delta_psi"])    # (n_ATSE,)

# coverage per cassette = cells covering its skip junction
skip_col_of = {}
for j in np.where(role_all == "J3")[0]:
    skip_col_of[col_ev[j]] = j
cov_atse = np.array([ (Cd[:, skip_col_of[i]] > 0).sum() for i in range(len(events)) ])

# a strong, well-covered differential cassette; and a well-covered null one
well = cov_atse >= 500
cand_diff = np.where(is_diff_atse & well)[0]
cand_null = np.where(~is_diff_atse & well)[0]
ai_diff = cand_diff[np.argmax(delta_true_all[cand_diff])]     # biggest planted effect
ai_null = cand_null[np.argmax(cov_atse[cand_null])]           # best covered null

def plot_cassette(ai, ax_row, title):
    """Per-cell PSI of J1/J2/J3 across true programs, with the planted truth overlaid."""
    cols = np.where(col_ev == ai)[0]
    order = {"J1": 0, "J2": 1, "J3": 2}
    cols = sorted(cols, key=lambda c: order[role_all[c]])
    for ax, c in zip(ax_row, cols):
        r = role_all[c]
        covered = Cd[:, c] > 0
        df = pd.DataFrame({"program": prog_all[covered], "PSI": Rd[covered, c]})
        sns.violinplot(data=df, x="program", y="PSI", ax=ax, inner=None,
                       cut=0, color="lightsteelblue", linewidth=0.8)
        sns.stripplot(data=df.sample(min(400, len(df)), random_state=0),
                      x="program", y="PSI", ax=ax, size=1.8, color="0.3", alpha=0.5)
        # planted truth: J3 -> psi_skip ; J1/J2 -> (1 - psi_skip)/2
        t = true_skip[:, ai] if r == "J3" else (1 - true_skip[:, ai]) / 2
        ax.scatter(range(len(t)), t, marker="D", s=45, color="crimson", zorder=10,
                   label="planted truth")
        ax.set_ylim(-0.05, 1.05)
        ax.set_title(f"{r}" + ("  (SKIP)" if r == "J3" else "  (inclusion)"), fontsize=10)
        ax.set_xlabel("true program")
        if c != cols[0]:
            ax.set_ylabel("")
    ax_row[0].set_ylabel("per-cell PSI")
    ax_row[0].legend(fontsize=7, loc="upper left")
    ax_row[1].text(0.5, 1.28, title, transform=ax_row[1].transAxes,
                   ha="center", fontsize=12, weight="bold")

fig, axes = plt.subplots(2, 3, figsize=(13, 7.4))
plot_cassette(ai_diff, axes[0],
              f"DIFFERENTIAL cassette  {events[ai_diff]}\n"
              f"planted Δψ = {delta_true_all[ai_diff]:.2f}   ({cov_atse[ai_diff]:,} cells covered)")
plot_cassette(ai_null, axes[1],
              f"NULL cassette  {events[ai_null]}\n"
              f"planted Δψ = {delta_true_all[ai_null]:.2f}   ({cov_atse[ai_null]:,} cells covered)")
plt.tight_layout(); plt.show()

# the planted skip usage per program, and what the counts actually gave back
print(f"DIFFERENTIAL cassette {events[ai_diff]}")
tab = pd.DataFrame({
    "planted skip psi": true_skip[:, ai_diff].round(3),
    "recovered (counts)": emp[:, ai_diff].round(3),
    "cells covering": ncell[:, ai_diff].astype(int),
}, index=[f"program {k}" for k in range(true_skip.shape[0])])
print(tab.to_string())
print(f"\nNULL cassette {events[ai_null]}  (planted psi identical across programs)")
tab0 = pd.DataFrame({
    "planted skip psi": true_skip[:, ai_null].round(3),
    "recovered (counts)": emp[:, ai_null].round(3),
    "cells covering": ncell[:, ai_null].astype(int),
}, index=[f"program {k}" for k in range(true_skip.shape[0])])
print(tab0.to_string())


## 2. PCA on centred junction usage ratio

`junc_ratio_centered` is already in the object (computed by the simulator with the pipeline's
`calculate_centered_psi`). It is sparse and already centred, so we use **TruncatedSVD**, which does not
re-centre and does not densify — the same approach `run_training.py` uses for its waypoint PCA.

In [ ]:
def pca_centered(adata, n_comps=20, seed=0):
    X = sp.csr_matrix(adata.layers["junc_ratio_centered"]).astype(np.float32)
    svd = TruncatedSVD(n_components=n_comps, random_state=seed)
    U = svd.fit_transform(X)
    return U * 1.0, svd.explained_variance_ratio_


pcs, var_ratio = pca_centered(a)
print(f"PC1 {100*var_ratio[0]:.1f}%  PC2 {100*var_ratio[1]:.1f}%  "
      f"top-10 cumulative {100*var_ratio[:10].sum():.1f}%")

fig, ax = plt.subplots(figsize=(4.5, 3))
ax.plot(np.arange(1, len(var_ratio) + 1), 100 * var_ratio, "o-")
ax.axvline(K, color="red", ls="--", lw=1, label=f"true K = {K}")
ax.set_xlabel("PC"); ax.set_ylabel("% variance"); ax.legend()
ax.set_title("Scree — an elbow near true K is expected")
plt.tight_layout(); plt.show()

In [ ]:
prog = a.obs["true_program"].values
ct = a.obs["broad_cell_type"].astype(str).values

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
for k in np.unique(prog):
    m = prog == k
    axes[0].scatter(pcs[m, 0], pcs[m, 1], s=6, alpha=0.6, label=f"program {k} (n={m.sum()})")
axes[0].set_xlabel(f"PC1 ({100*var_ratio[0]:.1f}%)")
axes[0].set_ylabel(f"PC2 ({100*var_ratio[1]:.1f}%)")
axes[0].set_title("Coloured by TRUE program\n(the ground-truth latent structure)")
axes[0].legend(fontsize=7, markerscale=2)

for c in pd.Series(ct).value_counts().index:
    m = ct == c
    axes[1].scatter(pcs[m, 0], pcs[m, 1], s=6, alpha=0.6, label=f"{c} (n={m.sum()})")
axes[1].set_xlabel(f"PC1 ({100*var_ratio[0]:.1f}%)")
axes[1].set_ylabel(f"PC2 ({100*var_ratio[1]:.1f}%)")
axes[1].set_title("Coloured by the real heart cell type\n(programs were seeded from these)")
axes[1].legend(fontsize=6, markerscale=2)
plt.tight_layout(); plt.show()

## 3. The null control — PCA must find *nothing*

The `effect=0.0` dataset has identical skip usage across programs for **every** cassette. It shares the
same real coverage, the same phi, the same cells. So any program separation PCA shows here is a **false
positive** produced by coverage structure or Binomial noise alone.

This is the single most important panel: it tells us that the separation seen above is real signal and
not an artefact of the coverage we conditioned on.

In [ ]:
b = ad.read_h5ad(NULL)
pcs_n, var_n = pca_centered(b)
prog_n = b.obs["true_program"].values

fig, axes = plt.subplots(1, 2, figsize=(11, 4.4), sharex=False)
for k in np.unique(prog):
    m = prog == k
    axes[0].scatter(pcs[m, 0], pcs[m, 1], s=6, alpha=0.6)
axes[0].set_title(f"SIGNAL (30% differential)\nPC1 {100*var_ratio[0]:.1f}%")
for k in np.unique(prog_n):
    m = prog_n == k
    axes[1].scatter(pcs_n[m, 0], pcs_n[m, 1], s=6, alpha=0.6)
axes[1].set_title(f"NULL (no cassette differs)\nPC1 {100*var_n[0]:.1f}%  — expect NO program structure")
for ax in axes:
    ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
plt.tight_layout(); plt.show()

In [ ]:
# Quantify "is there program structure?" — how well do the top PCs separate the true programs?
# silhouette on the leading PCs: ~0 = no structure, higher = programs are separable.
from sklearn.metrics import silhouette_score

for name, P, pr in [("SIGNAL", pcs, prog), ("NULL  ", pcs_n, prog_n)]:
    s = silhouette_score(P[:, :5], pr, sample_size=3000, random_state=0)
    print(f"{name}: silhouette of TRUE programs on PC1-5 = {s:+.3f}")
print("\nSIGNAL should be clearly positive; NULL should be ~0 (no recoverable program structure).")

## 4. Count / coverage realism

The simulation inherits ATSE coverage verbatim from real heart data, so depth and dropout should look
like the real thing rather than like a clean textbook simulation.

In [ ]:
C = sp.csr_matrix(a.layers["cell_by_cluster_matrix"])
J = sp.csr_matrix(a.layers["cell_by_junction_matrix"])
ratio = sp.csr_matrix(a.layers["junc_ratio"])

fig, ax = plt.subplots(1, 3, figsize=(13, 3.4))
ax[0].hist(np.log10(C.data + 1), bins=60, color="steelblue")
ax[0].set_xlabel("log10(ATSE coverage + 1)"); ax[0].set_ylabel("(cell, junction) entries")
ax[0].set_title("ATSE coverage — taken from REAL data")

ax[1].hist(ratio.data, bins=50, color="darkslateblue")
ax[1].set_xlabel("junction usage ratio (PSI)")
ax[1].set_title("Simulated PSI\n(U-shaped: Beta(0.5,0.5) prior on skip usage)")

cov_per_cell = np.asarray((C > 0).sum(1)).ravel()
ax[2].hist(cov_per_cell, bins=50, color="seagreen")
ax[2].set_xlabel("junctions covered per cell")
ax[2].set_title("Per-cell coverage (dropout is real)")
plt.tight_layout(); plt.show()

sparsity = 100 * (1 - C.nnz / (a.n_obs * a.n_vars))
print(f"matrix sparsity: {sparsity:.1f}% of (cell, junction) entries have ZERO coverage")
print(f"median ATSE coverage where covered: {np.median(C.data):.0f} reads")

---
# 2. Metrics — what "recovery" means

Defined **once** here and reused by every fit below, so every number in the sweep is comparable.

The programs are identifiable only **up to a permutation**, so everything is Hungarian-matched to the
truth before scoring.

| metric | what it answers | chance |
|---|---|---|
| `psi_r` | did we recover the **programs** (ψ)? Median matched correlation with true ψ | 0 |
| `auroc` | can the model tell a **differential** cassette from a **null** one? | 0.5 |
| `fdp@0.05`, `power` | is the **FDR actually controlled**? Realised false-discovery proportion at a nominal 5% | — |
| `agree` | did each cell get its **right program**? Dominant-program agreement | 1/K_true |
| `ari` | do the inferred φ **cluster the cells into the true programs**? Adjusted Rand Index | 0 |
| `psi_contrast` | *diagnostic.* Are the K programs different from each other, or collapsed? | ~0 = collapse |
| `bb_conc` | *diagnostic.* When ψ is wrong the Beta-Binomial collapses the concentration to soak up the residual | true value |
| `elbo` | the only thing available **without ground truth** — does it track recovery? (§6) | — |

`agree` and `ari` are the two that answer *"does φ recover the cell types"*. `auroc` + `fdp` answer
*"does it find the right differentially spliced junctions"*.

In [ ]:
# ── METRICS: defined once, used by every fit ─────────────────────────────────
import io, contextlib, itertools, os, time
import numpy as np, pandas as pd, torch
from scipy.optimize import linear_sum_assignment
from sklearn.metrics import roc_auc_score, adjusted_rand_score

# `sim` is the SIGNAL simulation -- cell 1 loaded it as `a`. Everything below fits and scores on it.
sim = a

def to_np(x):
    """Model outputs come back as torch tensors (possibly on GPU), numpy, or lists. Normalise."""
    return x.detach().cpu().numpy() if isinstance(x, torch.Tensor) else np.asarray(x)

def truth_of(sim):
    """Ground truth for a sim, aligned to the J3 (skip) junction of each cassette."""
    ev   = np.asarray(sim.uns["atse_ids"], dtype=object)
    ei   = {x: i for i, x in enumerate(ev)}
    cev  = np.array([ei[x] for x in sim.var["event_id"].astype(str).values])
    skp  = np.where(sim.var["junction_role"].values == "J3")[0]
    ordr = cev[skp]
    tp   = np.asarray(sim.uns["true_psi_skip"])[:, np.sort(np.unique(ordr))]   # (K_true, n_ATSE)
    isd  = np.array([sim.var["is_differential"].values[cev == i][0] for i in range(len(ev))])
    return dict(psi=tp, phi=np.asarray(sim.obsm["true_phi"]), skip_idx=skp, order=ordr,
                is_diff=isd, K_true=tp.shape[0])

def score_fit(T, psi_hat, phi_hat):
    """Score one fit against ground truth T. Works when K_fit != K_true."""
    tp, tphi, skp, ordr = T["psi"], T["phi"], T["skip_idx"], T["order"]
    Kt = T["K_true"]
    ph = psi_hat[:, skp][:, np.argsort(ordr)]           # learned skip usage, (K_fit, n_ATSE)
    Kf = ph.shape[0]

    # Hungarian match learned programs -> true programs
    C = np.zeros((Kf, Kt))
    for i in range(Kf):
        for j in range(Kt):
            m = np.isfinite(tp[j]) & np.isfinite(ph[i])
            C[i, j] = np.corrcoef(ph[i][m], tp[j][m])[0, 1] if m.sum() > 2 else 0.0
    r, c = linear_sum_assignment(-np.abs(np.nan_to_num(C)))
    psi_r = float(np.median([C[i, j] for i, j in zip(r, c)]))

    # PHI: did the cells get their right program?  ("cell-type recovery")
    agree = float((phi_hat[:, r].argmax(1) == tphi[:, c].argmax(1)).mean())
    ari   = float(adjusted_rand_score(tphi.argmax(1), phi_hat.argmax(1)))   # permutation-invariant

    # DS: can the model tell differential from null cassettes?
    d     = ph.max(0) - ph.min(0)                       # cross-program spread per cassette = effect size
    ok    = np.isfinite(d)
    auroc = float(roc_auc_score(T["is_diff"][ok], d[ok]))
    return dict(psi_r=psi_r, agree=agree, ari=ari, auroc=auroc,
                chance_agree=1.0 / Kt, effect=d, effect_ok=ok)

def crude_threshold_calibration(T, effect, ok, alpha=0.05, delta=0.05):
    """A CRUDE point-estimate threshold: call every cassette whose effect exceeds `delta`.

    *** THIS IS NOT THE MODEL'S FDR PROCEDURE. *** It uses no posterior and does no FDR control, so
    its "fdp" is NOT a statement about whether LeafletFA controls false positives -- it is just the
    error rate of a naive threshold. It is kept only as a cheap per-fit summary of how separable the
    differential and null cassettes are. The REAL FDR analysis is 6.3, which uses posterior samples
    and bayesian_fdr(). Do not quote these columns."""
    isd  = T["is_diff"][ok]
    eff  = effect[ok]
    called = eff > delta
    if called.sum() == 0:
        return dict(n_called=0, fdp=0.0, power=0.0)
    fdp   = float((~isd[called]).sum() / called.sum())      # realised FDP among the called
    power = float((isd[called]).sum() / max(isd.sum(), 1))
    return dict(n_called=int(called.sum()), fdp=fdp, power=power)

print("metrics defined: score_fit() -> psi_r, agree, ari, auroc ; crude_threshold_calibration() [NOT the FDR procedure -- see 6.3]")

---
# 3. The reference fit — does LeafletFA recover the truth at all?

Everything in the sweep is measured **relative to this configuration**, so it is worth looking at on
its own first. If the reference cannot recover a simulation generated from the model's own generative
process, nothing downstream is interpretable.

**Reference:** `K=5` (the true K), **Beta-Binomial**, `lr=0.05`, `gamma=0.5`, early stopping **off**,
random init, full-batch, `seed=0`.

In [ ]:
# ── the ONE fit everything else is compared against ──────────────────────────
from leafletfa import LeafletFA
from leafletfa.chunked import fit_chunked   # the SINGLE chunked implementation — see leafletfa/chunked.py

REF = dict(K=5, likelihood="bb", lr=0.05, junc_prior=True, waypoints=False,
           batching="full", seed=0)
EPOCHS = 1200

def fit_once(sim, K, likelihood, lr, junc_prior, waypoints, batching, seed, epochs=EPOCHS):
    """One LeafletFA fit. `patience` is effectively OFF: early stopping has twice silently
    truncated these fits (METHODS.md 11.3), so on the simulation we always run the full budget."""
    conc = torch.tensor(float("inf")) if likelihood == "binom" else None
    common = dict(K=K, junc_specific_prior=junc_prior, input_conc_prior=conc,
                  waypoints_use=waypoints, print_epochs=10**6,
                  patience=10**6, min_delta=0, min_delta_rel=1e-6,      # early stopping OFF
                  ELBO_num_particles=5, num_samples=50, lr=lr, gamma=0.5,
                  log_wandb=False, loss_plot=False)
    t0 = time.time()
    if batching == "full":
        m = LeafletFA(adata=sim, num_epochs=epochs, **common)
        with contextlib.redirect_stdout(io.StringIO()):
            m.from_anndata(); m.train(num_initializations=1, seeds=[seed]); m.get_all_variables()
    else:
        # CHUNKED: call the PACKAGED implementation. Do NOT hand-roll this loop here.
        #
        # This notebook used to re-implement the chunked scheme by hand, and it drifted: it dropped
        # the per-cell phi carry-over and invented a final frozen-psi E-step that production never
        # runs. So full-batch-vs-chunked -- the result that justifies chunking for the ENTIRE atlas
        # -- was comparing against a scheme nobody uses. leafletfa.chunked.fit_chunked() now exists
        # precisely so the benchmark and production cannot diverge. It carries phi per cell, keeps
        # the accumulated store as the final phi, and applies obs_scale = N/B.
        m = fit_chunked(sim, K=K, batch_size=2000, num_passes=6, seed=seed,
                        num_epochs_first=max(60, epochs // 10),
                        num_epochs_later=max(60, epochs // 20),
                        lr=lr, **{k: v for k, v in common.items() if k not in ("lr", "K")})
    return m, time.time() - t0

T = truth_of(sim)
m_ref, secs = fit_once(sim, **REF)
s = score_fit(T, to_np(m_ref.psi), to_np(m_ref.assign_post))
cal = crude_threshold_calibration(T, s["effect"], s["effect_ok"])

print(f"REFERENCE  K={REF['K']} {REF['likelihood']} lr={REF['lr']} "
      f"junc_prior={REF['junc_prior']} waypoints={REF['waypoints']} {REF['batching']}-batch  "
      f"({secs/60:.1f} min)\n")
print(f"  psi_r (program recovery)     {s['psi_r']:+.3f}     (chance 0)")
print(f"  agree (cells -> right prog)  {s['agree']:.1%}      (chance {s['chance_agree']:.0%})")
print(f"  ARI   (phi clusters cells)   {s['ari']:.3f}      (chance 0)")
print(f"  AUROC (differential vs null) {s['auroc']:.3f}      (chance 0.5)")
print(f"  [crude threshold, NOT the FDR procedure -- see 6.3] called={cal['n_called']}  realised FDP={cal['fdp']:.3f}  power={cal['power']:.3f}")
print(f"  psi_contrast                 {m_ref.health['psi_contrast']:.4f}  (~0.003 = collapse)")
print(f"  bb_conc                      {m_ref.bb_conc}")

---
# 4. The sweep — which model parameters actually matter?

**Design: one axis at a time from the reference, plus a full cross of the one pair that interacts.**

A full factorial over every axis would be `3 K x 2 likelihood x 4 lr x 2 prior x 2 init x 2 batching x
3 seeds` = **576 fits**. That buys almost nothing: most of those cells answer questions no one asked.
Varying one axis at a time from a fixed reference answers *"which parameter outcompetes"* in **39 fits**
and is far easier to read.

The one exception is **`lr` x `likelihood`**, which are **known to interact** — the Binomial is a much
sharper likelihood than the Beta-Binomial, so an `lr` tuned on one is not valid for the other. Those two
get a full cross.

**Every configuration is run with 3 seeds.** Seed variance on this model is large enough that a
single-seed comparison is meaningless, and reproducibility across seeds is itself one of the results
(§5).

| axis | levels | the question |
|---|---|---|
| `lr` x `likelihood` | 4 x 2, crossed | the schedule. Does the "wrong" likelihood really win? |
| `K` | 5, 10, 20 | behaviour when **over-specified** (the atlas K is not known a priori) |
| `junc_prior` | junction-specific / global | **the open question** — see below |
| `waypoints` | on / off | is waypoint init load-bearing, or does random init get there too? |
| `batching` | full / chunked | does the production minibatch scheme match full-batch? |

> **On `junc_prior`.** `ψ_kj ~ Beta(a_j, b_j)` indexes `a_j`/`b_j` by **junction only** — all K programs
> at a junction share one *learned* Beta. That is a shrinkage prior pulling the programs **toward each
> other**, i.e. toward the symmetric saddle that is this model's main failure mode. On a **small**
> synthetic sim (500 cells, 90 junctions) the global prior won decisively (ψ r 0.99 vs 0.31) and removed
> all seed variance. **But that sim is thin** — with 90 junctions, `a_j`/`b_j` are fit from almost no data
> and the prior dominates the likelihood. This sim is 6,986 x 12,804 and is the honest test. **If
> `junc_prior=True` does fine here, the small-sim result was a small-data artifact.** The paper uses
> `True` and it has never been tested. This cell is the test.

In [ ]:
# ── THE SWEEP ────────────────────────────────────────────────────────────────
# Cached to CSV: delete SWEEP_CSV to force a re-run. ~39 fits, ~2 h on an L40S.
SWEEP_CSV = "sweep_results.csv"
SEEDS     = [0, 1, 2]

configs = []
def add(tag, **kw):
    cfg = {**REF, **kw}
    for sd in SEEDS:
        configs.append(dict(axis=tag, **{**cfg, "seed": sd}))

# Block A — lr x likelihood, FULLY CROSSED (they interact)
for lk in ["bb", "binom"]:
    for lr in [0.005, 0.01, 0.02, 0.05]:
        add("lr x likelihood", likelihood=lk, lr=lr)

# Block B — one axis at a time from the reference
for K in [10, 20]:            add("K", K=K)                       # K=5 is the reference
add("psi prior",  junc_prior=False)                               # True is the reference
add("init",       waypoints=True)                                 # random is the reference
add("batching",   batching="chunked")                             # full is the reference

# de-duplicate: the reference appears in Block A already
seen, uniq = set(), []
for c in configs:
    key = tuple(sorted((k, v) for k, v in c.items() if k != "axis"))
    if key not in seen:
        seen.add(key); uniq.append(c)
configs = uniq
print(f"{len(configs)} fits ({len(configs)//len(SEEDS)} configs x {len(SEEDS)} seeds)")

# RESUMABLE + INCREMENTAL. Each fit is appended to the CSV as soon as it finishes, so:
#   * progress is visible while the job runs (tail the CSV),
#   * a crash at fit 38 of 39 does not throw away the other 38,
#   * re-running skips whatever is already done.
# Delete SWEEP_CSV to force a clean re-run.
FIT_KEY = ["K", "likelihood", "lr", "junc_prior", "waypoints", "batching", "seed"]

done = pd.read_csv(SWEEP_CSV) if os.path.exists(SWEEP_CSV) else pd.DataFrame(columns=FIT_KEY)
done_keys = {tuple(r) for r in done[FIT_KEY].values} if len(done) else set()
todo = [c for c in configs
        if tuple(c[k] for k in FIT_KEY) not in done_keys]
print(f"{len(done)} fits already done, {len(todo)} to run")

if todo:
    rows = []
    for n, cfg in enumerate(todo, 1):
        cfg = dict(cfg)
        axis = cfg.pop("axis")
        try:
            m, secs = fit_once(sim, **cfg)
            s   = score_fit(T, to_np(m.psi), to_np(m.assign_post))
            cal = crude_threshold_calibration(T, s["effect"], s["effect_ok"])
            cc  = m.bb_conc
            rows.append(dict(axis=axis, **cfg,
                psi_r=s["psi_r"], agree=s["agree"], ari=s["ari"], auroc=s["auroc"],
                chance_agree=s["chance_agree"], crude_fdp=cal["fdp"], crude_power=cal["power"],
                n_called=cal["n_called"], psi_contrast=m.health["psi_contrast"],
                bb_conc=(np.nan if isinstance(cc, str) else float(np.asarray(cc))),
                elbo=float(m.best_elbo), healthy=bool(m.health["healthy"]), minutes=secs / 60))
            print(f"[{n:>2}/{len(todo)}] {axis:<16} K={cfg['K']:<3} {cfg['likelihood']:<5} "
                  f"lr={cfg['lr']:<6} jp={str(cfg['junc_prior']):<5} wp={str(cfg['waypoints']):<5} "
                  f"{cfg['batching']:<7} s{cfg['seed']} -> psi_r={s['psi_r']:+.3f} "
                  f"ari={s['ari']:.3f} auroc={s['auroc']:.3f}", flush=True)
            # APPEND this fit immediately -- do not wait for the whole sweep
            pd.DataFrame([rows[-1]]).to_csv(
                SWEEP_CSV, mode="a", header=not os.path.exists(SWEEP_CSV), index=False)
        except Exception as e:
            print(f"[{n:>2}/{len(todo)}] FAILED {axis} {cfg}: {type(e).__name__}: {e}", flush=True)

sweep = pd.read_csv(SWEEP_CSV)
print(f"\n{len(sweep)} fits in {SWEEP_CSV}")

display(sweep.round(3))

---
# 5. Reproducibility across random seeds

SVI is criticised for landing in different local optima from different starts, and this model is a
genuinely hard case: it begins **on a symmetric saddle** (`AutoDiagonalNormal` initialises every cell's
φ at the prior median and all K programs at nearly the same ψ), so the K programs only separate once
gradient noise breaks the symmetry.

So "is it reproducible?" is not a formality here — it is a **result**. For every configuration in the
sweep, the seed spread is reported below. A configuration whose seeds disagree is not usable, **whatever
its best single run looks like.**

In [ ]:
# ── Seed variance, per configuration ─────────────────────────────────────────
key = ["axis", "K", "likelihood", "lr", "junc_prior", "waypoints", "batching"]
rep = (sweep.groupby(key)
            .agg(n=("seed", "size"),
                 psi_r_mean=("psi_r", "mean"), psi_r_sd=("psi_r", "std"),
                 psi_r_min=("psi_r", "min"),   psi_r_max=("psi_r", "max"),
                 ari_mean=("ari", "mean"),     ari_sd=("ari", "std"),
                 auroc_mean=("auroc", "mean"), auroc_sd=("auroc", "std"))
            .reset_index()
            .sort_values("psi_r_mean", ascending=False))
display(rep.round(3))

print("THE NUMBER THAT MATTERS: psi_r_sd — the run-to-run spread.")
print(f"  most reproducible : sd={rep.psi_r_sd.min():.3f}")
print(f"  least             : sd={rep.psi_r_sd.max():.3f}")
unstable = rep[rep.psi_r_sd > 0.10]
if len(unstable):
    print(f"\n  {len(unstable)} config(s) with psi_r_sd > 0.10 — NOT REPRODUCIBLE:")
    for _, r in unstable.iterrows():
        print(f"    {r.axis:<16} K={r.K} {r.likelihood} lr={r.lr} jp={r.junc_prior} "
              f"wp={r.waypoints} {r.batching}: psi_r {r.psi_r_min:+.3f}..{r.psi_r_max:+.3f}")
else:
    print("\n  no configuration has psi_r_sd > 0.10 — every arm is reproducible across seeds.")

# Seed spread visualised
fig, ax = plt.subplots(figsize=(9, max(4, 0.32 * len(rep))))
lbl = rep.apply(lambda r: f"{r.axis}: K{r.K} {r.likelihood} lr{r.lr} "
                          f"jp{str(r.junc_prior)[0]} wp{str(r.waypoints)[0]} {r.batching[:4]}", axis=1)
y = np.arange(len(rep))
ax.errorbar(rep.psi_r_mean, y, xerr=rep.psi_r_sd.fillna(0), fmt="o", color="#2b6cb0",
            ecolor="#a0aec0", capsize=3, ms=5)
ax.set_yticks(y); ax.set_yticklabels(lbl, fontsize=7)
ax.axvline(0, color="grey", lw=0.8, ls=":")
ax.set_xlabel("psi_r  (mean ± sd over 3 seeds)"); ax.set_ylabel("")
ax.set_title("Program recovery and its seed variance, per configuration")
ax.invert_yaxis(); plt.tight_layout(); plt.show()

---
# 6. Summary — which parameters clearly outcompete?

This is the section that goes in the paper. Four questions, in order:

1. **Which axis matters?** Effect of each parameter on program recovery (ψ), differential-splicing
   detection (AUROC) and cell-type recovery (φ / ARI).
2. **`lr` x `likelihood`** — the interaction, as a heatmap.
3. **Is the FDR actually controlled?** Realised false-discovery proportion vs nominal. *(Reviewer 1:
   "no clear empirical estimate of false positives".)*
4. **Does the ELBO track recovery?** On real data there is no ground truth, so model selection has to
   run on the ELBO. **If the ELBO does not correlate with ψ recovery here, then selecting on it in the
   atlas is unjustified** — and that would be a load-bearing negative result.

In [ ]:
# ── 6.1 WHICH AXIS MATTERS? ──────────────────────────────────────────────────
# CRITICAL: compare each axis ONLY against the REFERENCE setting of every other axis.
#
# A naive `sweep.groupby(axis).psi_r.mean()` is CONFOUNDED. The reference level of each axis
# (K=5, full-batch, junc_prior=True, ...) also appears all through the lr x likelihood block --
# including the terrible lr=0.005 fits -- so the reference level gets dragged down by runs that have
# nothing to do with that axis. That confound alone is enough to make K=10 "beat" the true K=5 and
# chunked "beat" full-batch. One-factor-at-a-time means holding the others FIXED.
METRICS = [("psi_r", "program recovery (psi_r)", 0.0),
           ("auroc", "differential vs null (AUROC)", 0.5),
           ("ari",   "cell-type recovery (phi ARI)", 0.0)]
AXES    = [("likelihood", "likelihood"), ("lr", "learning rate"), ("K", "K"),
           ("junc_prior", "psi prior"), ("waypoints", "waypoint init"), ("batching", "batching")]

def ofat(df, axis_col):
    """Rows in which ONLY `axis_col` departs from the reference."""
    others = [c for c, _ in AXES if c != axis_col]
    m = np.ones(len(df), bool)
    for c in others:
        m &= (df[c] == REF[c]).values
    return df[m]

fig, axes = plt.subplots(len(METRICS), len(AXES),
                         figsize=(3.0 * len(AXES), 2.7 * len(METRICS)), sharey="row")
for i, (mcol, mlab, chance) in enumerate(METRICS):
    for j, (acol, alab) in enumerate(AXES):
        ax  = axes[i, j]
        sub = ofat(sweep, acol)                       # <- the un-confounded comparison
        lv  = sorted(sub[acol].unique(), key=str)
        for x, level in enumerate(lv):
            v = sub[sub[acol] == level][mcol].dropna().values
            if len(v) == 0:
                continue
            ax.scatter(np.full(len(v), x) + np.random.uniform(-.06, .06, len(v)), v,
                       s=18, alpha=.6, color="#2b6cb0", zorder=3)
            ax.hlines(v.mean(), x - .25, x + .25, color="#c53030", lw=2, zorder=4)
        ax.axhline(chance, color="grey", ls=":", lw=.9)
        ax.set_xticks(range(len(lv)))
        ax.set_xticklabels([str(l) for l in lv], fontsize=7, rotation=30, ha="right")
        if i == 0:
            ax.set_title(f"{alab}\n(ref: {REF[acol]})", fontsize=8)
        if j == 0:
            ax.set_ylabel(mlab, fontsize=8)
        ax.tick_params(labelsize=7)
fig.suptitle("Which parameters matter?   one axis varied, ALL OTHERS HELD AT THE REFERENCE   "
             "|   point = one seed, red bar = mean, dotted = chance", fontsize=9)
plt.tight_layout(rect=[0, 0, 1, 0.95]); plt.show()

print("HOW MUCH EACH AXIS MOVES PROGRAM RECOVERY (psi_r), holding the others at the reference:")
print(f"  reference = {REF}\n")
imp = []
for acol, alab in AXES:
    g = ofat(sweep, acol).groupby(acol).psi_r.mean()
    if len(g) > 1:
        imp.append((alab, g.max() - g.min(), g.idxmax(), g.max(), g.idxmin(), g.min()))
for alab, spread, bl, bv, wl, wv in sorted(imp, key=lambda t: -t[1]):
    verdict = "MATTERS A LOT" if spread > 0.15 else ("matters" if spread > 0.05 else "barely matters")
    print(f"  {alab:<16} spread {spread:.3f}  best={str(bl):<7} ({bv:+.3f})   "
          f"worst={str(wl):<7} ({wv:+.3f})   {verdict}")
print("\nThe axes with a SMALL spread are the good news: they are the ones the result does NOT")
print("depend on. If `waypoint init` barely matters, then 'programs are recovered from random")
print("initialization; waypoints only converge faster' -- a far stronger answer to the local-optima")
print("objection than 'we needed waypoints'.")

In [ ]:
# ── 6.2 lr x likelihood — the interaction ────────────────────────────────────
blk = sweep[sweep.axis == "lr x likelihood"]
if len(blk):
    fig, axs = plt.subplots(1, 3, figsize=(13, 3.4))
    for ax, (mcol, mlab, _) in zip(axs, METRICS):
        piv = blk.pivot_table(index="likelihood", columns="lr", values=mcol, aggfunc="mean")
        im = ax.imshow(piv.values, cmap="viridis", aspect="auto")
        ax.set_xticks(range(piv.shape[1])); ax.set_xticklabels(piv.columns)
        ax.set_yticks(range(piv.shape[0])); ax.set_yticklabels(piv.index)
        for r in range(piv.shape[0]):
            for c in range(piv.shape[1]):
                v = piv.values[r, c]
                ax.text(c, r, f"{v:.2f}", ha="center", va="center", fontsize=8,
                        color="w" if v < np.nanmean(piv.values) else "k")
        ax.set_title(mlab, fontsize=9); ax.set_xlabel("learning rate")
        plt.colorbar(im, ax=ax, fraction=.046)
    fig.suptitle("lr x likelihood — these INTERACT: the Binomial is a sharper likelihood, "
                 "so an lr tuned on one is not valid for the other", fontsize=9)
    plt.tight_layout(rect=[0, 0, 1, 0.93]); plt.show()

    bb  = blk[blk.likelihood == "bb"].groupby("lr").psi_r.mean()
    bi  = blk[blk.likelihood == "binom"].groupby("lr").psi_r.mean()
    print("psi_r by lr:")
    for lr in sorted(blk.lr.unique()):
        print(f"  lr={lr:<7} Beta-Binomial {bb.get(lr, np.nan):+.3f}   Binomial {bi.get(lr, np.nan):+.3f}")
    print("\nNB the simulation was generated with a BINOMIAL emission. If Beta-Binomial still wins,")
    print("that is not a modelling error -- the learned concentration absorbs residual misfit and")
    print("makes the objective easier to descend. Check the `bb_conc` column.")

In [ ]:
# ── 6.3 IS THE FDR ACTUALLY CONTROLLED?  (Reviewer 1) ────────────────────────
#
# THIS RUNS THE MODEL'S REAL DECISION PROCEDURE, NOT A THRESHOLD ON THE POINT ESTIMATE.
#
# An earlier version of this cell called `effect > delta` a discovery. That is NOT the method -- it
# uses no posterior, no P(|effect| > delta), and no FDR control -- so its "realised FDP" (0.29) said
# nothing about whether the model controls false positives. The real procedure:
#
#   1. S posterior samples of psi                                   -> psi_samples [S, K, J]
#   2. per cassette e and sample s, the effect is the cross-program spread of the skip junction:
#          beta_se = max_k psi[s,k,e] - min_k psi[s,k,e]
#   3. P_e = (1/S) #{ s : beta_se > delta }                         <- posterior prob of a real effect
#   4. bayesian_fdr(P, alpha) keeps the MOST PERMISSIVE set whose posterior expected FDP <= alpha
#      (max detections; FDP(n) = running mean of (1 - P) over the top n)
#
# scored against the planted truth. The NULL simulation (effect = 0.0) is the one that matters:
# EVERY discovery there is a false positive by construction, so it is a direct empirical
# false-positive rate -- exactly what Reviewer 1 said the manuscript lacked.
from leafletfa.differential_splicing_groups import bayesian_fdr

DELTA, ALPHA = 0.10, 0.05

def cassette_probs(m, T_, delta=DELTA):
    """P(effect > delta) per cassette, from the POSTERIOR -- not from the point estimate."""
    # NB psi_samples only exists after prune_K(); the raw posterior is all_samples["psi"].
    ps   = to_np(m.all_samples["psi"])                                # [S, K, J]
    sk   = ps[:, :, T_["skip_idx"]][:, :, np.argsort(T_["order"])]   # [S, K, n_ATSE]
    beta = sk.max(1) - sk.min(1)                                     # [S, n_ATSE]
    return (beta > delta).mean(0)                                    # [n_ATSE]

def fdr_report(m, T_, label, alpha=ALPHA, delta=DELTA, quiet=False):
    P = cassette_probs(m, T_, delta)
    called, _fdp, n_sig = bayesian_fdr(P, alpha=alpha)   # NB returns (bool mask, fdp, n_sig)
    isd = T_["is_diff"]
    n   = int(called.sum())
    fp  = int((called & ~isd).sum())
    fdp = fp / n if n else 0.0
    pwr = int((called & isd).sum()) / max(int(isd.sum()), 1)
    if not quiet:
        print(label)
        print(f"   cassettes            {len(P)}      truly differential: {int(isd.sum())}")
        print(f"   CALLED at alpha={alpha}   {n}")
        print(f"   false positives      {fp}")
        print(f"   REALISED FDP         {fdp:.4f}   (nominal {alpha})")
        print(f"   power                {pwr:.4f}")
    return dict(n_called=n, fp=fp, fdp=fdp, power=pwr, P=P, called=called, n_tot=len(P))

# ---- SIGNAL sim: is the FDR controlled where there IS signal? ---------------------------
sig = fdr_report(m_ref, T, f"SIGNAL sim (effect=1.0)")

# ---- NULL sim: every call is a false positive BY CONSTRUCTION ---------------------------
b_null = ad.read_h5ad(NULL)
m_null, _ = fit_once(b_null, **REF)
T_null    = truth_of(b_null)
print()
nul = fdr_report(m_null, T_null, "NULL sim (effect=0.0 -- there is NOTHING to find)")
fpr = nul["n_called"] / nul["n_tot"]
print(f"\n   ===> EMPIRICAL FALSE-POSITIVE RATE = {nul['n_called']} / {nul['n_tot']} = {fpr:.4f}")

# ---- the operating curve --------------------------------------------------------------
fig, axs = plt.subplots(1, 2, figsize=(10, 3.6))
deltas = np.linspace(0.02, 0.30, 12)
f_sig, p_sig, f_nul = [], [], []
for dl in deltas:
    s_ = fdr_report(m_ref,  T,      "", delta=dl, quiet=True)
    n_ = fdr_report(m_null, T_null, "", delta=dl, quiet=True)
    f_sig.append(s_["fdp"]); p_sig.append(s_["power"])
    f_nul.append(n_["n_called"] / n_["n_tot"])

axs[0].plot(deltas, f_sig, "o-", color="#c53030", ms=4, label="realised FDP (signal sim)")
axs[0].plot(deltas, f_nul, "s-", color="#2b6cb0", ms=4, label="false-positive rate (NULL sim)")
axs[0].axhline(ALPHA, color="grey", ls="--", lw=1, label=f"nominal {ALPHA}")
axs[0].set_xlabel("effect-size threshold  delta"); axs[0].set_ylabel("false-discovery fraction")
axs[0].set_title("Is the FDR controlled?"); axs[0].legend(fontsize=7)
axs[1].plot(deltas, p_sig, "o-", color="#2b6cb0", ms=4)
axs[1].set_xlabel("effect-size threshold  delta"); axs[1].set_ylabel("power")
axs[1].set_title("...and what it costs in power")
plt.tight_layout(); plt.show()

print("\nThis is the empirical false-positive estimate Reviewer 1 asked for. LEAD WITH THE NULL-SIM")
print("NUMBER. Do NOT argue from the posterior width -- it is not characterised, and its error is not")
print("even consistent in sign under chunking (METHODS.md 11.4c).")

In [ ]:
# ── 6.4 DOES THE ELBO TRACK RECOVERY?  (can we select without ground truth?) ──
from scipy.stats import spearmanr
ok = sweep.dropna(subset=["elbo", "psi_r"])
fig, axs = plt.subplots(1, 2, figsize=(10, 3.8))

axs[0].scatter(ok.elbo, ok.psi_r, s=22, alpha=.7, c="#2b6cb0")
rho_all, p_all = spearmanr(ok.elbo, ok.psi_r)
axs[0].set_xlabel("ELBO (loss — LOWER is better)"); axs[0].set_ylabel("psi_r")
axs[0].set_title(f"ACROSS configurations: rho={rho_all:+.2f} (p={p_all:.1g})", fontsize=9)

# The question that actually matters: WITHIN a config, does best-ELBO pick the best seed?
picks = []
for cfg, g in ok.groupby(key):
    if len(g) < 2: continue
    picks.append(dict(elbo_pick=g.loc[g.elbo.idxmin()].psi_r,   # LOWEST loss
                      best=g.psi_r.max(), worst=g.psi_r.min()))
pk = pd.DataFrame(picks)
if len(pk):
    axs[1].scatter(pk.best, pk.elbo_pick, s=30, c="#2b6cb0")
    lim = [min(pk.worst.min(), pk.elbo_pick.min()) - .05, 1.02]
    axs[1].plot(lim, lim, "k--", lw=1, label="perfect selection")
    axs[1].set_xlim(lim); axs[1].set_ylim(lim)
    axs[1].set_xlabel("psi_r of the BEST seed"); axs[1].set_ylabel("psi_r of the seed the ELBO PICKED")
    axs[1].set_title("WITHIN a config: does best-ELBO pick the best seed?", fontsize=9)
    axs[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

print("WHY THIS IS THE MOST IMPORTANT PLOT IN THE NOTEBOOK")
print("  On the atlas there is NO ground truth. Model selection HAS to run on the ELBO (or on a")
print("  held-out metric). This asks whether that is even legitimate.")
if len(pk):
    regret = (pk.best - pk.elbo_pick)
    print(f"\n  ELBO-selection regret (best psi_r - selected psi_r): "
          f"mean {regret.mean():.3f}, worst {regret.max():.3f}")
    if regret.max() < 0.05:
        print("  -> the ELBO is a SAFE selector here. Selecting on it in the atlas is justified.")
    else:
        print("  -> the ELBO does NOT reliably pick the best seed. Do NOT select on it in the atlas;")
        print("     use a HELD-OUT metric (07c cell-type decodability / 04* imputation) instead.")

---
## What to carry into the Methods

Fill this in from the tables above once the sweep has run. The claims that need to survive review:

1. **The simulation is a valid benchmark** (§1) — realistic coverage, real noise, a genuine null in
   which PCA finds nothing.
2. **LeafletFA recovers the planted programs** (§3) — ψ, the differential cassettes, and the cell-type
   structure in φ, all far above chance.
3. **Which parameters matter, and which do not** (§6.1) — this is the honest version of "we tuned it".
4. **Programs are recovered from random initialization** (§6.1, `waypoint init` panel). If random init
   matches waypoints, say so — *"waypoints converge faster; they are not required"* is a far stronger
   answer to the local-optima objection than *"we needed waypoints"*.
5. **The result is reproducible across seeds** (§5).
6. **The FDR is empirically controlled** (§6.3) — Reviewer 1's request, answered with a number.
7. **Whether the ELBO can be used to select a model** (§6.4) — this licenses (or forbids) the selection
   strategy used on the atlas.